In [ ]:
import torch # Import main PyTorch library
import torch.nn as nn # Import neural network modules
import torch.optim as optim # Import optimization algorithms
import torchvision # Import computer vision library
import torchvision.transforms as transforms # Import image transformation tools
from torch.utils.data import DataLoader, random_split # Import data handling utilities
import matplotlib.pyplot as plt # Import plotting library
import numpy as np # Import numerical processing library
import time # Import time tracking library
import random # Import random number generator
import os # Import operating system utilities

def set_seed(seed=0):
    # Set seed for Python's built-in random module
    random.seed(seed)
    # Set seed for NumPy's random module
    np.random.seed(seed)
    # Set seed for PyTorch's CPU random generator
    torch.manual_seed(seed)
    # Check if a GPU is available
    if torch.cuda.is_available():
        # Set seed for current GPU
        torch.cuda.manual_seed(seed)
        # Set seed for all available GPUs
        torch.cuda.manual_seed_all(seed)
        # Force cuDNN to use deterministic algorithms
        torch.backends.cudnn.deterministic = True
        # Disable cuDNN benchmarking to ensure reproducibility
        torch.backends.cudnn.benchmark = False

# Execute seed initialization function
set_seed(0)

# Assign hardware device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_dataloaders(batch_size=64, val_split=0.2):
    # Define a pipeline for image transformations
    transform = transforms.Compose([
        # Convert image to PyTorch tensor format
        transforms.ToTensor(),
        # Normalize RGB channel values to the range [-1.0, 1.0]
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    # Download and load the full CIFAR-10 training dataset
    full_train_dataset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform
    )

    # Download and load the CIFAR-10 test dataset
    test_dataset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform
    )

    # Get total number of training samples
    num_train = len(full_train_dataset)
    # Calculate number of validation samples based on split ratio
    val_size = int(num_train * val_split)
    # Calculate remaining training samples
    train_size = num_train - val_size

    # Randomly split training dataset into train and validation sets
    train_dataset, val_dataset = random_split(
        full_train_dataset,[train_size, val_size],
        # Use fixed seed generator for consistent splits
        generator=torch.Generator().manual_seed(0)
    )

    # Create data loader for training set, enabling shuffling
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    # Create data loader for validation set, no shuffling
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    # Create data loader for test set, no shuffling
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Return loaders and dataset size integers
    return train_loader, val_loader, test_loader, train_size, val_size, len(test_dataset)

class SoftmaxRegression(nn.Module):
    # Define Softmax Regression initialization
    def __init__(self):
        # Initialize parent nn.Module class
        super(SoftmaxRegression, self).__init__()
        # Define linear layer mapping 3072 flattened inputs to 10 class outputs
        self.linear = nn.Linear(3 * 32 * 32, 10)

    # Define forward pass
    def forward(self, x):
        # Flatten image tensor from 2D (Batch, 3, 32, 32) into 1D (Batch, 3072)
        x = torch.flatten(x, 1)
        # Pass flattened data through the linear layer
        return self.linear(x)


class PlainCNN(nn.Module):
    # Define Plain CNN initialization taking number of convolutional layers
    def __init__(self, num_convs):
        # Initialize parent nn.Module class
        super(PlainCNN, self).__init__()

        # Initialize empty list to hold convolutional layers
        layers =[]
        # Set initial input channels to 3 (for RGB)
        in_channels = 3
        # Set output channels for each convolution to 16
        out_channels = 16

        # Loop to create the specified number of convolutional layers
        for i in range(num_convs):
            # Add a 2D convolution layer with 3x3 kernel and padding 1
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
            # Add a ReLU activation function layer
            layers.append(nn.ReLU())
            # Update input channels for the next layer
            in_channels = out_channels

        # Add a final Max Pooling layer to reduce spatial dimensions by half
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))

        # Package the layer list into a Sequential module
        self.features = nn.Sequential(*layers)

        # Define fully connected classification layers
        self.classifier = nn.Sequential(
            # Linear layer mapping pooled features (16*16*16) to 128 nodes
            nn.Linear(out_channels * 16 * 16, 128),
            # ReLU activation
            nn.ReLU(),
            # Output linear layer mapping 128 nodes to 10 classes
            nn.Linear(128, 10)
        )

    # Define forward pass
    def forward(self, x):
        # Pass input through feature extraction layers
        x = self.features(x)
        # Flatten the feature maps into a 1D vector
        x = torch.flatten(x, 1)
        # Pass flattened vector through the classifier
        x = self.classifier(x)
        # Return model predictions
        return x


class ResidualBlock(nn.Module):
    # Define Residual Block initialization
    def __init__(self, channels, dropout_prob=0.3):
        # Initialize parent nn.Module class
        super(ResidualBlock, self).__init__()
        # First 3x3 convolutional layer
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        # First batch normalization layer
        self.bn1 = nn.BatchNorm2d(channels)
        # Second 3x3 convolutional layer
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        # Second batch normalization layer
        self.bn2 = nn.BatchNorm2d(channels)
        # Dropout layer to prevent overfitting
        self.dropout = nn.Dropout(dropout_prob)

    # Define forward pass
    def forward(self, x):
        # Store original input to use for the skip connection
        residual = x

        # Pass input through first conv layer
        out = self.conv1(x)
        # Normalize the output
        out = self.bn1(out)
        # Apply ReLU activation
        out = torch.relu(out)
        # Apply dropout
        out = self.dropout(out)

        # Pass through second conv layer
        out = self.conv2(out)
        # Normalize the output
        out = self.bn2(out)

        # Add the original input (skip connection) to the output
        out += residual
        # Apply final ReLU activation and return
        return torch.relu(out)


class AdvancedCNN(nn.Module):
    # Define Advanced CNN initialization
    def __init__(self, num_convs, dropout_prob=0.3):
        # Initialize parent nn.Module class
        super(AdvancedCNN, self).__init__()

        # Define initial convolutional layer setup
        self.init_conv = nn.Sequential(
            # 3x3 convolution mapping 3 input channels to 16
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            # Normalize batch
            nn.BatchNorm2d(16),
            # ReLU activation
            nn.ReLU()
        )

        # Calculate number of residual blocks needed
        num_blocks = (num_convs - 1) // 2
        # Calculate if a standalone convolution is needed for an odd layer count
        remainder = (num_convs - 1) % 2

        # Initialize list to hold residual blocks
        blocks =[]
        # Loop to create residual blocks
        for _ in range(num_blocks):
            # Append a ResidualBlock to the list
            blocks.append(ResidualBlock(16, dropout_prob))

        # Package residual blocks into a Sequential module
        self.res_blocks = nn.Sequential(*blocks)

        # Check if an extra convolutional layer is required
        if remainder > 0:
            # Create extra convolution layer with batch norm and ReLU
            self.extra_conv = nn.Sequential(
                nn.Conv2d(16, 16, kernel_size=3, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU()
            )
        else:
            # Use a passthrough identity layer if not required
            self.extra_conv = nn.Identity()

        # Define 2x2 max pooling layer
        self.pool = nn.MaxPool2d(2, 2)

        # Define fully connected classification layers
        self.classifier = nn.Sequential(
            # Apply dropout
            nn.Dropout(dropout_prob),
            # Linear layer mapping pooled features to 128 nodes
            nn.Linear(16 * 16 * 16, 128),
            # ReLU activation
            nn.ReLU(),
            # Apply dropout
            nn.Dropout(dropout_prob),
            # Output linear layer mapping 128 nodes to 10 classes
            nn.Linear(128, 10)
        )

    # Define forward pass
    def forward(self, x):
        # Pass input through initial conv layer
        x = self.init_conv(x)
        # Pass through residual blocks
        x = self.res_blocks(x)
        # Pass through optional extra conv layer
        x = self.extra_conv(x)
        # Apply max pooling
        x = self.pool(x)
        # Flatten the tensors for fully connected layers
        x = torch.flatten(x, 1)
        # Pass through classifier
        x = self.classifier(x)
        # Return model predictions
        return x


def evaluate(model, dataloader, criterion):
    # Switch model to evaluation mode (disables dropout/batch norm updates)
    model.eval()
    # Initialize total loss accumulator
    total_loss = 0.0
    # Initialize correct predictions counter
    correct = 0
    # Initialize total samples counter
    total = 0

    # Disable gradient calculation for faster processing and lower memory use
    with torch.no_grad():
        # Iterate over batches in the dataloader
        for inputs, targets in dataloader:
            # Move inputs and targets to the active device
            inputs, targets = inputs.to(device), targets.to(device)
            # Get model predictions
            outputs = model(inputs)
            # Calculate batch loss
            loss = criterion(outputs, targets)

            # Accumulate total loss (scaled by batch size)
            total_loss += loss.item() * inputs.size(0)
            # Find the predicted class indices
            _, predicted = outputs.max(1)
            # Accumulate total number of samples
            total += targets.size(0)
            # Accumulate number of correct predictions
            correct += predicted.eq(targets).sum().item()

    # Calculate percentage accuracy
    acc = 100.0 * correct / total
    # Calculate average loss per sample
    avg_loss = total_loss / total
    # Return average loss and accuracy
    return avg_loss, acc

def train_model(model, model_name, train_loader, val_loader, test_loader,
                epochs=20, lr=0.001, batch_size=64, weight_decay=0.0,
                val_ratio=0.2, print_header=True):
    # Move model to the active hardware device
    model = model.to(device)

    # Initialize Cross Entropy loss function
    criterion = nn.CrossEntropyLoss()
    # Initialize Adam optimizer with given learning rate and weight decay
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Check if header output is enabled
    if print_header:
        # Print decorative header and training configuration details
        print("="*80)
        print("CIFAR-10 NEURAL NETWORK TRAINING")
        print("="*80)
        print(f"Model: {model_name}")
        print(f"Input size: 3072")
        print(f"Output size: 10")
        print(f"Epochs: {epochs}")
        print(f"Learning rate: {lr}")
        print(f"Batch size: {batch_size}")
        print(f"Data directory: ./data")
        print(f"Validation ratio: {val_ratio}")
        print(f"Random seed: 0")
        print("="*80)
        print("Loading CIFAR-10 dataset...")

        # Get sizes of each dataset split
        train_samples = len(train_loader.dataset)
        val_samples = len(val_loader.dataset)
        test_samples = len(test_loader.dataset)

        # Print dataset and label shapes
        print(f"Training data shape: ({train_samples}, 3072), Labels shape: ({train_samples},)")
        print(f"Test data shape: ({test_samples}, 3072), Labels shape: ({test_samples},)")
        print("Number of classes: 10")
        print("\nCreating validation split...")
        print("Final split:")
        print(f"  Training set: {train_samples} samples")
        print(f"  Validation set: {val_samples} samples")
        print(f"\nCreated {model_name} Classifier")
        print(f"\nStarting training for {epochs} epochs...")

    # Initialize dictionary to track metrics across epochs
    history = {'train_loss':[], 'val_loss':[], 'train_acc': [], 'val_acc':[]}

    # Track the highest validation accuracy seen so far
    best_val_acc = 0.0
    # Record overall start time
    start_time_total = time.time()

    # Loop over the number of epochs
    for epoch in range(epochs):
        # Switch model to training mode
        model.train()
        # Record start time for the current epoch
        start_time_epoch = time.time()

        # Initialize epoch training loss
        train_loss = 0.0
        # Initialize epoch correct predictions counter
        correct = 0
        # Initialize epoch total samples counter
        total = 0

        # Iterate over batches in the training dataloader
        for inputs, targets in train_loader:
            # Move batch data to the active device
            inputs, targets = inputs.to(device), targets.to(device)

            # Reset optimizer gradients to zero
            optimizer.zero_grad()

            # Compute model outputs for current batch
            outputs = model(inputs)

            # Calculate the loss for the batch
            loss = criterion(outputs, targets)

            # Compute gradients via backpropagation
            loss.backward()

            # Update model weights based on computed gradients
            optimizer.step()

            # Accumulate loss (scaled by batch size)
            train_loss += loss.item() * inputs.size(0)
            # Find the predicted classes
            _, predicted = outputs.max(1)
            # Accumulate total sample count
            total += targets.size(0)
            # Accumulate correct prediction count
            correct += predicted.eq(targets).sum().item()

        # Calculate average training loss for the epoch
        train_loss = train_loss / total
        # Calculate training accuracy percentage
        train_acc = 100.0 * correct / total

        # Evaluate model on the validation set
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        # Evaluate model on the test set
        test_loss, test_acc = evaluate(model, test_loader, criterion)
        # Calculate percentage error (100% - Accuracy%)
        test_error = 100.0 - test_acc

        # Calculate time taken for the epoch
        epoch_time = time.time() - start_time_epoch

        # Store epoch metrics in history dictionary
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # Print formatted string of epoch metrics
        print(f"Epoch {epoch+1:>3}/{epochs} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Acc: {val_acc:.2f}% | Test Loss: {test_loss:.4f} | "
              f"Test Acc: {test_acc:.2f}% | Test Error: {test_error:.2f}% | "
              f"Time: {epoch_time:.2f}s")

        # Check if validation accuracy improved
        if val_acc > best_val_acc:
            # Update best validation accuracy
            best_val_acc = val_acc
            # Print improvement notification
            print("        *** New best validation accuracy! ***")

    # Calculate total training time
    total_time = time.time() - start_time_total
    # Print total training time
    print(f"\nTraining completed in {total_time:.2f}s")

    # Evaluate final model on train dataset
    final_train_loss, final_train_acc = evaluate(model, train_loader, criterion)
    final_train_error = 100.0 - final_train_acc

    # Evaluate final model on validation dataset
    final_val_loss, final_val_acc = evaluate(model, val_loader, criterion)
    final_val_error = 100.0 - final_val_acc

    # Evaluate final model on test dataset
    final_test_loss, final_test_acc = evaluate(model, test_loader, criterion)
    final_test_error = 100.0 - final_test_acc

    # Formatted final output for Train, Validation, and Test
    print(f"Final Train Loss: {final_train_loss:.4f} | Final Train Acc: {final_train_acc:.2f}% | Final Train Error: {final_train_error:.2f}%")
    print(f"Final Val Loss:   {final_val_loss:.4f} | Final Val Acc:   {final_val_acc:.2f}% | Final Val Error:   {final_val_error:.2f}%")
    print(f"Final Test Loss:  {final_test_loss:.4f} | Final Test Acc:  {final_test_acc:.2f}% | Final Test Error:  {final_test_error:.2f}%")

    # Return final test accuracy, total time, and metric history
    return final_test_acc, total_time, history

def plot_curves(history_dict, title):
    # Generate list of epoch numbers for the x-axis
    epochs = range(1, len(history_dict['train_loss']) + 1)

    # Initialize a new figure with specific dimensions
    plt.figure(figsize=(12, 5))

    # Create first subplot for plotting Loss
    plt.subplot(1, 2, 1)
    # Plot training loss
    plt.plot(epochs, history_dict['train_loss'], label='Train Loss')
    # Plot validation loss
    plt.plot(epochs, history_dict['val_loss'], label='Val Loss')
    # Set subplot title
    plt.title(f'{title} - Loss')
    # Label x-axis
    plt.xlabel('Epochs')
    # Label y-axis
    plt.ylabel('Loss')
    # Add a legend
    plt.legend()
    # Enable grid lines
    plt.grid(True)

    # Create second subplot for plotting Accuracy
    plt.subplot(1, 2, 2)
    # Plot training accuracy
    plt.plot(epochs, history_dict['train_acc'], label='Train Acc')
    # Plot validation accuracy
    plt.plot(epochs, history_dict['val_acc'], label='Val Acc')
    # Set subplot title
    plt.title(f'{title} - Accuracy')
    # Label x-axis
    plt.xlabel('Epochs')
    # Label y-axis
    plt.ylabel('Accuracy (%)')
    # Add a legend
    plt.legend()
    # Enable grid lines
    plt.grid(True)

    # Adjust layout to prevent overlap
    plt.tight_layout()
    # Display the plots
    plt.show()


def run_experiment_a(train_loader, val_loader, test_loader):
    # Print section separator for Experiment A
    print("\n" + "="*50)
    # Print experiment title
    print("RUNNING EXPERIMENT A: CNN Depth Analysis")
    # Print section separator
    print("="*50)

    # Define depths to test
    depths = [2, 8, 16, 32]
    # Initialize results dictionary
    results = {}

    # Loop over depths to train Plain CNNs
    for depth in depths:
        # Define model name based on depth
        model_name = f"PlainCNN_{depth}_Layers"
        # Print initialization status
        print(f"\n--- Initializing {model_name} ---")
        # Instantiate PlainCNN model
        model = PlainCNN(num_convs=depth)

        # Train PlainCNN model and retrieve outputs
        test_acc, time_taken, history = train_model(
            model, model_name, train_loader, val_loader, test_loader,
            epochs=15, lr=0.001
        )
        # Store results for current model
        results[model_name] = {'test_acc': test_acc, 'history': history}
        # Plot training curves for current model
        plot_curves(history, model_name)

    # Define subset of deep layer counts for Advanced CNN testing
    deep_depths =[16, 32]
    # Loop over deeper layer sizes
    for depth in deep_depths:
        # Define model name for advanced version
        model_name = f"AdvancedCNN_{depth}_Layers"
        # Print initialization status
        print(f"\n--- Initializing {model_name} with Residuals & Weight Decay ---")
        # Instantiate AdvancedCNN model
        model = AdvancedCNN(num_convs=depth, dropout_prob=0.3)

        # Train AdvancedCNN using weight decay
        test_acc, time_taken, history = train_model(
            model, model_name, train_loader, val_loader, test_loader,
            epochs=15, lr=0.001, weight_decay=1e-4
        )
        # Store results for advanced model
        results[model_name] = {'test_acc': test_acc, 'history': history}
        # Plot training curves for advanced model
        plot_curves(history, model_name)


def run_experiment_b(train_loader, val_loader, test_loader):
    # Print section separator for Experiment B
    print("\n" + "="*50)
    # Print experiment title
    print("RUNNING EXPERIMENT B: Learning Rate Analysis")
    # Print section separator
    print("="*50)

    # Define learning rates to test
    lrs =[0.000001, 0.0001, 0.001, 0.01, 1.0]
    # Initialize results dictionary
    results = {}

    # Loop through each learning rate
    for lr in lrs:
        # Define model name
        model_name = f"PlainCNN_8_Layers_LR_{lr}"
        # Print current learning rate status
        print(f"\n--- Training with Learning Rate: {lr} ---")
        # Instantiate fresh PlainCNN model for fair comparison
        model = PlainCNN(num_convs=8)

        # Train the model using the current learning rate
        test_acc, time_taken, history = train_model(
            model, model_name, train_loader, val_loader, test_loader,
            epochs=10, lr=lr
        )
        # Save results mapped by learning rate string
        results[str(lr)] = {'test_acc': test_acc, 'history': history}
        # Plot training curves
        plot_curves(history, model_name)


def run_experiment_c(val_split=0.2):
    # Print section separator for Experiment C
    print("\n" + "="*50)
    # Print experiment title
    print("RUNNING EXPERIMENT C: Mini-batch Size Study")
    # Print section separator
    print("="*50)

    # Define batch sizes to test
    batch_sizes =[1, 8, 16, 64, 256]
    # Initialize results dictionary
    results = {}

    # Limit epochs to keep runtime manageable
    epochs = 5

    # Loop over each batch size
    for bs in batch_sizes:
        # Print data preparation status for the current batch size
        print(f"\n--- Preparing Data with Batch Size: {bs} ---")
        # Regenerate dataloaders with the specific batch size
        tr_ld, v_ld, te_ld, _, _, _ = get_dataloaders(batch_size=bs, val_split=val_split)

        # Define model name reflecting current batch size
        model_name = f"PlainCNN_8_Layers_BS_{bs}"
        # Instantiate fresh PlainCNN model
        model = PlainCNN(num_convs=8)

        # Train the model with the active batch size loader
        test_acc, time_taken, history = train_model(
            model, model_name, tr_ld, v_ld, te_ld,
            epochs=epochs, lr=0.001, batch_size=bs
        )

        # Store accuracy, time, and history
        results[str(bs)] = {
            'test_acc': test_acc,
            'time_taken': time_taken,
            'history': history
        }

        # Print summary for current batch size iteration
        print(f"Batch Size {bs} -> Time: {time_taken:.2f}s, Final Test Acc: {test_acc:.2f}%")



if __name__ == '__main__':
    # Print the active device (CPU or GPU)
    print(f"Using device: {device}")

    # Load standardized datasets with batch size 64 for initial experiments
    train_loader, val_loader, test_loader, tr_size, v_size, te_size = get_dataloaders(batch_size=64)

    # Run Experiment A on depth analysis
    run_experiment_a(train_loader, val_loader, test_loader)
    # Run Experiment B on learning rates
    run_experiment_b(train_loader, val_loader, test_loader)
    # Run Experiment C on mini-batch sizes
    run_experiment_c(val_split=0.2)

    # Print starting status for baseline regression sanity check
    print("\n--- Testing Softmax Regression (Sanity Check) ---")
    # Instantiate Softmax Regression baseline model
    sr_model = SoftmaxRegression()
    # Train the baseline regression model
    train_model(sr_model, "SoftmaxRegression", train_loader, val_loader, test_loader, epochs=5, lr=0.001)